# Niural support agent - v2

OpenAI GPT-5.4 Mini handles support queries; GPT-5.4 is the optional correctness judge.
Four account tools: one constrained query tool and three detail lookups.
Pinned model versions are loaded from this version's `.env`.
Use **Streaming chat** for conversation, or run one selected benchmark question. Judging is optional.
No automatic benchmark loop, retries, manifests, or tracing.


## 1. Setup
Select the **Conda base** kernel (`C:/Users/Nirvana/miniconda3/python.exe`). `OPENAI_API_KEY` is loaded from the project `.env`.
This cell creates the agent without making a model request.


In [36]:
from pathlib import Path
import json
import sys
from IPython.display import display, Markdown

repo = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "mock_api/data").is_dir() and (p / "niural_support_agent/config.py").is_file())
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from niural_support_agent.config import Config, BENCHMARK_DIR
from mock_api import MockAPI, APIError
from niural_support_agent.agent import SupportAgent
from niural_support_agent.evaluate import evaluate_one

config = Config.load()
agent = SupportAgent(config)
display({"provider": config.provider, "model": config.model, "judge_model": config.judge_model})


{'provider': 'openai',
 'model': 'gpt-5.4-mini-2026-03-17',
 'judge_model': 'openai/gpt-oss-120b'}

## 2. Mock API and token auth
`mock_api/` contains the client, auth, and Pydantic data models. Any nonempty
mock token is accepted; a missing token raises a 401 API error. This token is separate from the Groq key.


In [37]:
api = MockAPI(token=config.mock_token)
employees = api.list_employees()
display({"employee_records": len(employees.data), "source": employees.source.model_dump(mode="json")})

try:
    MockAPI(token="").list_vendors()
except APIError as error:
    display(error.as_dict())

display([tool.name for tool in agent.agent.tools])


{'employee_records': 21,
 'source': {'reference': 'GET /employer/emplr-4f2a91c8/employees',
  'path': 'mock_api/data/employees.json',
  'as_of': '2026-09-28'}}

{'error': {'status': 401, 'message': 'An admin token is required.'}}

['query_account', 'get_employee', 'get_vendor_invoice', 'get_payment_status']

### Generated function schema (no model request)
The SDK creates this definition from the function signature and docstring. All four tools are available upfront; no deferred loading.


In [38]:
tool = next(t for t in agent.agent.tools)
display({"type": "function", "name": tool.name, "description": tool.description,
         "parameters": tool.params_json_schema, "strict": tool.strict_json_schema})


{'type': 'function',
 'name': 'query_account',
 'description': 'Filter, select, count or sum authenticated account data using Python.\n\nUse this for every account list, name lookup, headcount and balance.\nemployees fields: employee_id, first_name, last_name, status,\nemployment_type, work_location.city/state/country_iso2 (use dotted paths).\nActive means status=ONBOARDED; EMPLOYEE excludes US_1099_CONTRACTOR.\nvendors fields: vendor_id, name, default_currency.\ninvoices fields: invoice_id, invoice_number, vendor_id, vendor_name,\namount, currency, status, due_date, paid_date.\nStatuses and worker types use exact uppercase API values. Text contains\nis case-insensitive; other comparisons are exact. Filters use AND.\nFor outstanding balance exclude PAID, including pending/scheduled bills.\nSum invoice amount with Decimal; currencies are always kept separate.\nEmpty select/groups means no matches, not a tool failure. No pagination.\nReturn IDs in selected fields when a follow-up detail 

### Local query example (no model request)
This exercises the same validated query and Python executor used by the tool.
Counts come from code; the model only chooses the query and explains its result.


In [39]:
from niural_support_agent.query import AccountQuery, query_records

query = AccountQuery(
    resource="employees", operation="count",
    filters=[{"field": "status", "operator": "eq", "value": "ONBOARDED"}],
    group_by=["employment_type"],
)
display(query_records(api.list_employees(), query).model_dump(mode="json"))


{'query': {'resource': 'employees',
  'operation': 'count',
  'filters': [{'field': 'status', 'operator': 'eq', 'value': 'ONBOARDED'}],
  'fields': [],
  'group_by': ['employment_type'],
  'sum_field': None},
 'matched_count': 16,
 'records': [],
 'aggregates': [{'group': {'employment_type': 'EMPLOYEE'}, 'value': 14},
  {'group': {'employment_type': 'US_1099_CONTRACTOR'}, 'value': 2}],
 'source': {'reference': 'GET /employer/emplr-4f2a91c8/employees',
  'path': 'mock_api/data/employees.json',
  'as_of': '2026-09-28'}}

## 3. Streaming chat
Run this cell, then type a message. Each message starts one agent turn; text appears as it arrives.
`/reset` clears this chat, `/quit` exits. No judge calls are made.


In [ ]:
chat_session = "notebook-chat"
print("Chat ready. /reset clears history; /quit exits.")

while True:
    try:
        question = input("You: ").strip()
    except EOFError:
        break
    if question.lower() == "/quit":
        break
    if question.lower() == "/reset":
        agent.reset(chat_session)
        print("Conversation cleared.")
        continue
    if not question:
        continue

    print("Assistant: ", end="", flush=True)
    chunks = []
    def show_text(text):
        chunks.append(text)
        print(text, end="", flush=True)

    response = await agent.ask(chat_session, question, on_text=show_text)
    if response.error:
        print("\n" + (response.error.get("detail") or response.error["message"]))
    elif not chunks:
        print(response.answer, end="")
    print("\n")


Chat ready. /reset clears history; /quit exits.


## 4. Select one benchmark question
Change `case_id` to any integer from 1 to 8. Reference facts are used only by the optional judge.


In [ ]:
cases = json.loads((BENCHMARK_DIR / "cases.json").read_text(encoding="utf-8"))
case_id = 1
case = next(row for row in cases if row["id"] == case_id)
print(case["question"])


How many active employees do we have?


## 5. Generate one benchmark answer
Set `RUN_ANSWER = True` when ready. One question may require several model/tool calls
(up to four model turns). A rate-limit error stops the run; there are no automatic retries.


In [ ]:
RUN_ANSWER = True
result = None
if RUN_ANSWER:
    result = await evaluate_one(agent, case)
    display(Markdown(result["answer"]) if result["answer"] else result["error"])
    display(result["usage"])
else:
    print("Answer generation is off. Set RUN_ANSWER = True to run this question.")


You have **16 active workers** total as of **2026-09-28**:

- **14 employees**
- **2 contractors**

Source: `GET /employer/emplr-4f2a91c8/employees`

{'requests': 2,
 'input_tokens': 14440,
 'output_tokens': 229,
 'total_tokens': 14669}

## 6. Optional correctness judge
This makes an additional model request. Leave it off to save quota.
The answer is compared with the verified reference facts using the shared rubric.


In [ ]:
RUN_JUDGE = True
if RUN_JUDGE and result and result.get("answer") and result["id"] == case["id"]:
    verdict, usage = await agent.judge(
        case["question"], result["context"], case["facts"], result["answer"])
    display(verdict)
    display(usage)
else:
    print("Judge is off. Enable it explicitly after generating an answer.")


NotFoundError: Error code: 404 - The model `openai/gpt-oss-120b` does not exist or you do not have access to it.

## 6. Run all eight in LangSmith (optional)
Restart the kernel and rerun setup after enabling LangSmith in v2/.env.
Set RUN_ALL to True only when ready. This creates/reuses the dataset and runs eight answers plus one judge per successful answer.
Default concurrency is one. Set judge=False to save answers/traces without paid judge calls.


In [ ]:
from niural_support_agent.run_benchmark import run_benchmark

RUN_ALL = False
if RUN_ALL:
    experiment = await run_benchmark(
        agent, name="account-tools",
        description="Hypothesis: typed aggregation prevents counting errors",
        max_concurrency=1, judge=True,
    )
    display(experiment.to_pandas())
else:
    print("Batch evaluation is off. Set RUN_ALL = True to run all eight cases.")


## 7. V2 observations

- What worked:
- What failed:
- One proposed change:
- Retain / rollback / inconclusive:

Results remain in notebook outputs. Nothing is automatically saved to run folders.
